# Train a Model for Detecting Surface Water

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/train_water_detection.ipynb)

## Install package
To use the `geoai-py` package, ensure it is installed in your environment. Uncomment the command below if needed.

In [ ]:
%pip install geoai-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
  

## Import libraries

In [3]:
import geoai

ModuleNotFoundError: No module named 'geoai'

## Download sample data

In [1]:
train_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_train.tif"
train_masks_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_masks.tif"
test_raster_url = "/content/sentinel2_bands/sentinel2_all_bands.tif"

In [2]:
train_raster_path = geoai.download_file(train_raster_url)
train_masks_path = geoai.download_file(train_masks_url)
test_raster_path = geoai.download_file(test_raster_url)

NameError: name 'geoai' is not defined

In [4]:
geoai.print_raster_info(train_raster_path, show_preview=False)

===== RASTER INFORMATION: naip_water_train.tif =====
Driver: GTiff
Dimensions: 5390 x 7580 pixels
Number of bands: 4
Data type: uint8
Coordinate Reference System: EPSG:26914
Georeferenced Bounds: BoundingBox(left=485430.0, bottom=5204860.0, right=490820.0, top=5212440.0)
Pixel Resolution: 1.0, 1.0
NoData Value: None

----- Band Statistics -----
Band 1:
  Min: 36.00
  Max: 238.00
  Mean: 127.01
  Std Dev: 34.38
Band 2:
  Min: 46.00
  Max: 233.00
  Mean: 121.56
  Std Dev: 22.03
Band 3:
  Min: 57.00
  Max: 237.00
  Mean: 105.59
  Std Dev: 18.37
Band 4:
  Min: 19.00
  Max: 238.00
  Mean: 135.77
  Std Dev: 36.26


## Visualize sample data

In [5]:
geoai.view_raster(train_masks_url, nodata=0, basemap=train_raster_url)

In [6]:
geoai.view_raster(test_raster_url)

## Create training data

In [7]:
out_folder = "output"

In [8]:
tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_masks_path,
    tile_size=512,
    stride=128,
    buffer_radius=0,
)

Detected in_class_data as raster: naip_water_masks.tif
Raster CRS: EPSG:26914
Raster dimensions: 5420 x 7602

Raster info for naip_water_train.tif:
  CRS: EPSG:26914
  Dimensions: 5390 x 7580
  Resolution: (1.0, 1.0)
  Bands: 4
  Bounds: BoundingBox(left=485430.0, bottom=5204860.0, right=490820.0, top=5212440.0)
Found 1 unique classes in raster: [1]


Generated: 2280, With features: 2170: 100%|██████████| 2280/2280 [03:04<00:00, 12.33it/s]


------- Export Summary -------
Total tiles exported: 2280
Tiles with features: 2170 (95.2%)
Average feature pixels per tile: 101525.1
Output saved to: output

------- Georeference Verification -------


## Train object detection model

In [15]:
geoai.train_MaskRCNN_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/models",
    num_channels=4,
    pretrained=True,
    batch_size=4,
    num_epochs=1,
    learning_rate=0.005,
    val_split=0.2,
)

Using device: cuda
Found 2280 image files and 2280 label files
Training on 1824 images, validating on 456 images
Epoch: 0, Batch: 0/456, Loss: 5.5054, Time: 1.98s
Epoch: 0, Batch: 10/456, Loss: 1.5048, Time: 9.92s
Epoch: 0, Batch: 20/456, Loss: 0.6333, Time: 9.74s
Epoch: 0, Batch: 30/456, Loss: 1.3209, Time: 9.77s
Epoch: 0, Batch: 40/456, Loss: 1.0961, Time: 10.08s
Epoch: 0, Batch: 50/456, Loss: 1.0308, Time: 10.38s
Epoch: 0, Batch: 60/456, Loss: 1.0807, Time: 10.38s
Epoch: 0, Batch: 70/456, Loss: 0.7910, Time: 10.10s
Epoch: 0, Batch: 80/456, Loss: 0.5828, Time: 10.19s
Epoch: 0, Batch: 90/456, Loss: 0.8323, Time: 10.20s
Epoch: 0, Batch: 100/456, Loss: 0.7380, Time: 10.13s
Epoch: 0, Batch: 110/456, Loss: 0.7885, Time: 10.20s
Epoch: 0, Batch: 120/456, Loss: 0.6799, Time: 10.24s
Epoch: 0, Batch: 130/456, Loss: 0.8675, Time: 10.53s
Epoch: 0, Batch: 140/456, Loss: 0.2859, Time: 10.21s
Epoch: 0, Batch: 150/456, Loss: 1.0210, Time: 10.16s
Epoch: 0, Batch: 160/456, Loss: 0.6815, Time: 10.31s
E

## Run inference

In [30]:
masks_path = "naip_water_prediction.tif"
model_path = f"{out_folder}/models/best_model.pth"

In [31]:
geoai.object_detection(
    test_raster_path,
    masks_path,
    model_path,
    window_size=512,
    overlap=128,
    confidence_threshold=0.3,
    batch_size=4,
    num_channels=4,
)

Processing 280 windows with size 512x512 and overlap 128...


315it [00:37,  8.41it/s]


Inference completed in 37.99 seconds
Saved prediction to naip_water_prediction.tif


## Vectorize masks

In [32]:
output_path = "naip_water_prediction.geojson"
gdf = geoai.raster_to_vector(
    masks_path, output_path, min_area=1000, simplify_tolerance=1
)

Vectorized data saved to naip_water_prediction.geojson


In [33]:
gdf = geoai.add_geometric_properties(gdf)

In [34]:
len(gdf)

79

In [35]:
geoai.view_vector_interactive(gdf, tiles=test_raster_url)

Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

In [ ]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

# Randomly generated dataset of parking violations-
# Define the number of rows
num_rows = 1000000

states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking",
              "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

# Create a date range
start_date = "2022-01-01"
end_date = "2022-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Which parking violation is most commonly committed by vehicles from various U.S states?

(df[["Registration State", "Violation Description"]]  # get only these two columns
 .value_counts()  # get the count of offences per state and per type of offence
 .groupby("Registration State")  # group by state
 .head(1)  # get the first row in each group (the type of offence with the largest count)
 .sort_index()  # sort by state name
 .reset_index()
)

In [ ]:
gdf["elongation"].hist()

In [ ]:
gdf_filtered = gdf[gdf["elongation"] < 10]

In [ ]:
len(gdf_filtered)

## Visualize results

In [13]:
geoai.view_vector_interactive(gdf_filtered, tiles=test_raster_url)

NameError: name 'gdf_filtered' is not defined

In [ ]:
geoai.create_split_map(
    left_layer=gdf_filtered,
    right_layer=test_raster_url,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_raster_url,
)

![image](https://github.com/user-attachments/assets/a269b5a0-9f72-4ed8-8b2d-a175bbc45a23)